# 06_mllm / chain_of_thought.ipynb

**Purpose:** Run the same 500 pairs from `mllm_inference.ipynb` through a **structured chain-of-thought (CoT) prompt** and compare against the free-form explanations.

**Why two passes?**
- Free-form (Notebook 06.2) gives natural prose — easy to read, but verdicts can be vague.
- Chain-of-thought forces the LLM to enumerate concrete checks (entities, location, time, source) before deciding. This usually improves verdict accuracy and gives us more attributable signals to cite per claim.

**What we report:**
- Step-wise CoT trace: ENTITIES, LOCATION, TIME, EVIDENCE, VERDICT.
- Per-step parse rates (how often Gemini actually filled each step).
- CoT vs free-form verdict accuracy on the same pairs.
- Confusion matrix of CoT-MLLM verdicts against GSAF predictions and ground truth.

**Inputs:**
- `cache/mllm/explanations_freeform.csv` (must exist — produced by `mllm_inference.ipynb`)
- Local images, manifest, Gemini API key (same as before).

**Outputs:**
- `cache/mllm/explanations_cot.csv`
- `cache/mllm/explanations_cot.jsonl`
- `paper/tables/mllm_freeform_vs_cot.csv` — head-to-head accuracy
- `paper/tables/mllm_cot_step_parse_rates.csv` — % of valid CoT steps
- `paper/figures/mllm_freeform_vs_cot.png` — accuracy bar chart
- `paper/figures/mllm_cot_confusion.png` — confusion matrix vs ground truth


In [ ]:
# Imports & path setup
from __future__ import annotations
import json, sys, time, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110

NB_DIR = Path.cwd()
ROOT = NB_DIR
for parent in [NB_DIR] + list(NB_DIR.parents):
    if (parent / "src" / "utils" / "paths.py").exists():
        ROOT = parent
        break
sys.path.insert(0, str(ROOT))

from src.utils.paths import (
    ALL_MANIFEST, FUSION_CACHE, PROCESSED_IMAGES, VERITE_IMAGES,
    PAPER, TABLES, FIGURES,
)

CACHE_MLLM = ROOT / "cache" / "mllm"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# Verify dependencies
sentinel = CACHE_MLLM / ".gemini_setup_ok"
freeform_path = CACHE_MLLM / "explanations_freeform.csv"
assert sentinel.exists(), "Run 06_mllm/gemini_setup.ipynb first"
assert freeform_path.exists(), "Run 06_mllm/mllm_inference.ipynb first"

freeform = pd.read_csv(freeform_path)
print(f"Loaded {len(freeform)} free-form explanations.")
print(freeform.columns.tolist())


## 1. Re-resolve image paths (we need them for CoT calls)

In [ ]:
manifest = pd.read_csv(ALL_MANIFEST, usecols=["pair_id", "caption", "image_path"])
sample = freeform.merge(manifest.drop_duplicates("pair_id"), on="pair_id", how="left",
                        suffixes=("", "_mf"))

def resolve_image(p):
    if pd.isna(p) or not p: return None
    pp = Path(str(p))
    if pp.is_absolute() and pp.exists(): return pp
    for base in [PROCESSED_IMAGES, VERITE_IMAGES, ROOT / "data" / "raw" / "mmfakebench", ROOT]:
        cand = base / pp.name
        if cand.exists(): return cand
        cand2 = base / pp
        if cand2.exists(): return cand2
    return None

sample["_image_path"] = sample["image_path"].map(resolve_image)
print("Pairs with image:", sample["_image_path"].notna().sum(), "/", len(sample))


## 2. Chain-of-thought prompt

In [ ]:
COT_PROMPT = '''You are a fact-checking analyst. You are given an image and a caption.

Caption:
"""{caption}"""

Walk through the following five checks IN ORDER. Each line must start with the label and a colon, then exactly one short sentence. Do not add any extra commentary.

ENTITIES: Name the people, organisations, or objects you can identify in the image. If none recognisable, say "none recognised".
LOCATION: Describe the setting (indoors/outdoors, country/region if hinted by signs, vegetation, architecture).
TIME: Estimate the time of day, season, or era from visual cues (lighting, clothing, technology). If unclear, say "unclear".
EVIDENCE: State the single strongest reason the image either supports OR contradicts the caption.
VERDICT: Exactly one word — MATCH or MISMATCH.

Be concrete. Avoid hedging unless an item is genuinely unreadable.'''

print(COT_PROMPT.format(caption="<EXAMPLE>")[:600])


## 3. Run CoT calls (cached, restartable)

In [ ]:
from src.utils.gemini_client import GeminiClient

client = GeminiClient(rpm_limit=9)
print("Model:", client.model_name)

OUT_CSV   = CACHE_MLLM / "explanations_cot.csv"
OUT_JSONL = CACHE_MLLM / "explanations_cot.jsonl"

done_ids = set()
if OUT_CSV.exists():
    prev = pd.read_csv(OUT_CSV)
    done_ids = set(prev["pair_id"].astype(str).tolist())
    print(f"Already processed: {len(done_ids)}")

records = []
errors = []
work = sample[sample["_image_path"].notna()].copy()

for _, row in tqdm(work.iterrows(), total=len(work), desc="CoT calls"):
    pid = str(row["pair_id"])
    if pid in done_ids: continue
    cap = str(row["caption"])
    img = row["_image_path"]
    prompt = COT_PROMPT.format(caption=cap[:600])
    try:
        text = client.generate(prompt=prompt, image_path=img,
                               use_cache=True, temperature=0.1, max_output_tokens=400)
        rec = {
            "pair_id":  pid,
            "dataset":  row["dataset"],
            "bucket":   row["bucket"],
            "y_true":   int(row["y_true"]),
            "y_pred":   int(row["y_pred"]),
            "fusion_prob": float(row["fusion_prob"]),
            "caption":  cap,
            "cot_text": text,
        }
        records.append(rec)
        with open(OUT_JSONL, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    except Exception as e:
        errors.append({"pair_id": pid, "error": str(e)})
        if records:
            pd.DataFrame(records).to_csv(OUT_CSV, index=False)

# Combine
if OUT_CSV.exists():
    prev = pd.read_csv(OUT_CSV)
    new_df = pd.DataFrame(records)
    cot = pd.concat([prev, new_df], ignore_index=True).drop_duplicates("pair_id", keep="last")
else:
    cot = pd.DataFrame(records)
cot.to_csv(OUT_CSV, index=False)
print(f"\nTotal CoT explanations: {len(cot)}")
print(f"Errors: {len(errors)}")


## 4. Parse CoT steps

In [ ]:
STEP_LABELS = ["ENTITIES", "LOCATION", "TIME", "EVIDENCE", "VERDICT"]

def parse_steps(text):
    out = {k: None for k in STEP_LABELS}
    if not isinstance(text, str): return out
    for label in STEP_LABELS:
        m = re.search(rf"^\s*{label}\s*:\s*(.+)$", text, re.IGNORECASE | re.MULTILINE)
        if m:
            out[label] = m.group(1).strip()
    return out

def parse_verdict(text):
    m = re.search(r"VERDICT\s*:\s*(MATCH|MISMATCH)", text, re.IGNORECASE)
    if m: return m.group(1).upper()
    for line in reversed(str(text or "").strip().splitlines()):
        ln = line.strip().upper()
        if "MISMATCH" in ln: return "MISMATCH"
        if "MATCH" in ln: return "MATCH"
    return None

cot = pd.read_csv(OUT_CSV)
parsed = cot["cot_text"].map(parse_steps).apply(pd.Series)
parsed.columns = [f"step_{c.lower()}" for c in parsed.columns]
cot = pd.concat([cot, parsed], axis=1)
cot["cot_verdict"] = cot["cot_text"].map(parse_verdict)
cot["cot_pred_ooc"] = cot["cot_verdict"].map({"MISMATCH": 1, "MATCH": 0})
cot.to_csv(OUT_CSV, index=False)

print("Step-wise parse rates (% non-null):")
parse_rates = (parsed.notna().mean() * 100).round(1)
print(parse_rates)
parse_rates.to_csv(TABLES / "mllm_cot_step_parse_rates.csv", header=["pct_parsed"])
print("Saved:", TABLES / "mllm_cot_step_parse_rates.csv")


## 5. Free-form vs CoT vs GSAF accuracy

In [ ]:
# Bring in free-form verdict
ff = freeform[["pair_id", "explanation"]].copy()
ff["ff_verdict"] = ff["explanation"].map(parse_verdict)
ff["ff_pred_ooc"] = ff["ff_verdict"].map({"MISMATCH": 1, "MATCH": 0})

both = cot.merge(ff[["pair_id", "ff_verdict", "ff_pred_ooc"]], on="pair_id", how="left")

rows = []
for ds_name in ["all"] + list(both["dataset"].unique()):
    d = both if ds_name == "all" else both[both["dataset"] == ds_name]
    n = len(d)
    if n == 0: continue
    gsaf_acc = (d["y_pred"] == d["y_true"]).mean()
    ff_mask  = d["ff_pred_ooc"].notna()
    cot_mask = d["cot_pred_ooc"].notna()
    ff_acc   = (d.loc[ff_mask,  "ff_pred_ooc"]  == d.loc[ff_mask,  "y_true"]).mean() if ff_mask.any() else np.nan
    cot_acc  = (d.loc[cot_mask, "cot_pred_ooc"] == d.loc[cot_mask, "y_true"]).mean() if cot_mask.any() else np.nan
    rows.append({
        "dataset":  ds_name,
        "n":        n,
        "gsaf_acc": gsaf_acc,
        "ff_acc":   ff_acc,
        "cot_acc":  cot_acc,
        "ff_n":     int(ff_mask.sum()),
        "cot_n":    int(cot_mask.sum()),
    })

vs_df = pd.DataFrame(rows)
vs_df.to_csv(TABLES / "mllm_freeform_vs_cot.csv", index=False, float_format="%.4f")
print("Saved:", TABLES / "mllm_freeform_vs_cot.csv")
print()
print(vs_df.to_string(index=False))


## 6. Bar plot: accuracy comparison

In [ ]:
plot = vs_df.set_index("dataset")[["gsaf_acc", "ff_acc", "cot_acc"]].copy()
plot.columns = ["GSAF (model)", "MLLM free-form", "MLLM CoT"]
fig, ax = plt.subplots(figsize=(max(7, 1.4 * len(plot)), 5))
plot.plot(kind="bar", ax=ax, width=0.78)
ax.set_ylabel("Accuracy")
ax.set_title("Verdict accuracy on the 500-pair sample (per dataset)")
ax.set_ylim(0, 1.0)
ax.grid(axis="y", alpha=0.3)
ax.set_xticklabels(plot.index, rotation=0)
ax.legend(loc="lower right")
plt.tight_layout()
out = FIGURES / "mllm_freeform_vs_cot.png"
plt.savefig(out, bbox_inches="tight", dpi=150)
print("Saved:", out)
plt.show()


## 7. CoT confusion matrix vs ground truth

In [ ]:
mask = cot["cot_pred_ooc"].notna()
y_true = cot.loc[mask, "y_true"].astype(int).values
y_pred = cot.loc[mask, "cot_pred_ooc"].astype(int).values

if len(y_true) > 0:
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["MATCH (0)", "MISMATCH (1)"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["genuine (0)", "OOC (1)"])
    ax.set_xlabel("MLLM CoT verdict")
    ax.set_ylabel("Ground truth")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=12)
    ax.set_title(f"MLLM (CoT) confusion (n={mask.sum()})")
    plt.tight_layout()
    out = FIGURES / "mllm_cot_confusion.png"
    plt.savefig(out, bbox_inches="tight", dpi=150)
    print("Saved:", out)
    plt.show()

    print()
    print(classification_report(y_true, y_pred, target_names=["genuine (0)", "OOC (1)"], digits=4, zero_division=0))
else:
    print("No parseable CoT verdicts to score.")


## 8. Disagreement analysis: where CoT disagrees with GSAF

In [ ]:
disagree = cot[(cot["cot_pred_ooc"].notna()) & (cot["cot_pred_ooc"] != cot["y_pred"])]
print(f"CoT disagrees with GSAF on {len(disagree)} / {len(cot)} pairs")
print()
print("By dataset:")
print(disagree.groupby("dataset").size())
print()
print("Of disagreements, who is closer to ground truth?")
m = disagree["cot_pred_ooc"].notna()
cot_right  = (disagree.loc[m, "cot_pred_ooc"] == disagree.loc[m, "y_true"]).sum()
gsaf_right = (disagree.loc[m, "y_pred"]       == disagree.loc[m, "y_true"]).sum()
print(f"  CoT correct, GSAF wrong: {cot_right}")
print(f"  GSAF correct, CoT wrong: {gsaf_right}")

print("\nSample disagreements (CoT thinks OOC, GSAF says genuine, ground truth = OOC):")
samp = disagree[(disagree["cot_pred_ooc"] == 1) & (disagree["y_pred"] == 0) & (disagree["y_true"] == 1)].head(5)
for _, r in samp.iterrows():
    print(f"\n  pair_id: {r['pair_id']}  dataset: {r['dataset']}")
    print(f"  caption: {str(r['caption'])[:140]}")
    print(f"  CoT EVIDENCE: {str(r.get('step_evidence',''))[:160]}")


---
**Done.** This completes 06_mllm. The paper's MLLM section can now cite:
1. The 500-pair stratified sample.
2. CoT vs free-form verdict accuracy.
3. Per-step parse rates (showing the LLM does fill in entity / location / time / evidence reliably).
4. Cases where MLLM and GSAF disagree — useful for the limitations section.